In [7]:
import pathlib, os, sys, operator, re, datetime
from functools import reduce
import numpy as np
import tensorflow_datasets as tfds
import tensorflow as tf
from tiny_imagenet import TinyImagenetDataset
import matplotlib.pyplot as plt
from pathlib import Path


# model_path = os.path.abspath(""/home/<NETID>/path/to/your/lab1/CNN_TinyImageNet.h5)" # Uncomment this to use a non-relative path
model_path = os.path.abspath("CNN_TinyImageNet.h5")
model = tf.keras.models.load_model(model_path)

layer_outputs = [layer.output for layer in model.layers]
layer_inputs = [layer.input for layer in model.layers]
feature_map_model = tf.keras.Model(inputs=model.inputs, outputs=layer_outputs)

### build image datasets ###
tiny_imagenet_builder = TinyImagenetDataset()
tiny_imagenet_builder.download_and_prepare(download_dir="~/tensorflow-datasets/downloads")

ds = tiny_imagenet_builder.as_dataset()
ds_train, ds_val = ds["train"], ds["validation"]
assert(isinstance(ds_train, tf.data.Dataset))
assert(isinstance(ds_val, tf.data.Dataset))
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)
ds_val = ds_val.prefetch(tf.data.AUTOTUNE)

2025-10-29 23:21:49.568732: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-29 23:21:50.566450: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-29 23:21:51.034942: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-29 23:21:51.038303: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-29 23:21:51.771842: I tensorflow/core/platform/cpu_feature_gua

In [8]:
# We need to read the "human readable" labels so we can translate with the numeric values
# Read the labels file (wnids.txt)
with open(os.path.abspath('wnids.txt'), 'r') as f:
    wnids = [x.strip() for x in f]

# Map wnids to integer labels
wnid_to_label = {wnid: i for i, wnid in enumerate(wnids)}
label_to_wnid = {v: k for k, v in wnid_to_label.items()}

# Use words.txt to get names for each class
with open(os.path.abspath('words.txt'), 'r') as f:
    wnid_to_words = dict(line.split('\t') for line in f)
    for wnid, words in wnid_to_words.items():
        wnid_to_words[wnid] = [w.strip() for w in words.split(',')]

class_names = [str(wnid_to_words[wnid]) for wnid in wnids]

val_data = ds_val.take(1000).batch(1)

# non quantized prediction
for batch in ds_val.batch(1000).take(1):
    all_activations = feature_map_model.predict(tf.cast(batch["image"], tf.float32) / 255.0)
    print("FUCK")

# build validation dataset
val_imgs = []
for index, img_data in enumerate(val_data):
  val_imgs.append(img_data)


print(f"Total validation images collected: {len(val_imgs)}")

2025-10-29 23:22:01.922318: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 90ms/step
FUCK
Total validation images collected: 1000


2025-10-29 23:22:05.823455: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-10-29 23:22:05.921533: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2025-10-29 23:22:05.922315: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [ ]:
##########################################
# Helper Functions
##########################################

bits = 8  # Change this for different quantization ranges: 2, 4, or 8

if bits not in [2, 4, 8]:
    raise ValueError("Unsupported QUANT_BITS (must be 2, 4, or 8)")

# Signed for weights
min_int_w = - (1 << (bits - 1))
max_int_w = (1 << (bits - 1)) - 1

print(f"Weight quantization range: {min_int_w} to {max_int_w}")

# Unsigned for activations/inputs
min_int_a = - (1 << (bits - 1))
max_int_a = (1 << (bits-1)) - 1

print(f"Activation quantization range: {min_int_a} to {max_int_a}")

def plot_histogram(data, title, filename, save_dir, bins=100, data_range=None):
    """Plots and saves a histogram of the given data."""
    plt.figure(figsize=(10, 6))
    flat_data = data.flatten()
    
    # Determine range automatically if not provided
    if data_range is None:
        data_min, data_max = np.min(flat_data), np.max(flat_data)
        data_range = (data_min, data_max)
        # If min/max are too close, give a small buffer
        if data_range[0] == data_range[1]:
            data_range = (data_range[0] - 1, data_range[1] + 1)

    plt.hist(flat_data, bins=bins, range=data_range)
    plt.title(title)
    plt.xlabel("Quantized Value")
    plt.ylabel("Frequency")
    plt.grid(True)
    full_path = os.path.join(save_dir, filename + ".png")
    plt.savefig(full_path)
    plt.close() # Close the figure to save memory
    print(f"Saved plot: {full_path}")


Weight quantization range: -8 to 7
Activation quantization range: -8 to 7


In [22]:
###########################################
# Getters for Quantization Parameters #
############################################

def get_average(flat_input):
    """ Get the average value of the input """
    sum = 0
    for input in flat_input:
        sum += input
    average = sum / len(flat_input)
    return average

def get_max_difference(flat_input, average):
    """ Get the maximum absolute difference from the average value """
    max_diff = 0.0
    for input in flat_input:
        diff = abs(input - average)
        if diff > max_diff:
            max_diff = diff
    return max_diff

def get_max_weight(weights):
    """ Get the maximum absolute weight value """
    max_weight = 0
    flat_weights = weights.flatten()
    for weight in flat_weights:
        abs_weight = abs(weight)
        if abs_weight > max_weight:
            max_weight = abs_weight
    return max_weight

def get_input_zero_point(scale, avgIx):
    """ Get the input zero point """
    zp = -1 * round(scale * (avgIx))
    return zp

def get_input_scale(maxIx):
    """ Get the input scale """
    scale = max_int_a / maxIx
    return scale

def get_weight_scale(maxIw):
    """ Get the weight scale """
    scale = max_int_w / maxIw
    return scale

def get_bias_scale(scale_input, scale_weight):
    """ Get the bias scale """
    scale_bias = scale_input * scale_weight
    return scale_bias


In [23]:
##########################################
# Computational Functions #
##########################################

def quantize_inputs(inputs, scale, zero_point):
    """ Quantize the input activations """
    quantized = np.round(inputs * scale) + zero_point
    quantized = np.clip(quantized, min_int_a, max_int_a).astype(np.int8)
    return quantized

def quantize_weights(weights, scale):
    """ Quantize the weights """
    quantized = np.round(weights * scale)
    quantized = np.clip(quantized, min_int_w, max_int_w).astype(np.int8) # SHOULD NEVER EXCEED ANYWAYS
    return quantized

def compute_wi_zp_sum(weights, zero_point):
    """ Compute the sum of weights times input zero point per output channel """
    sums = np.sum(weights, axis=tuple(range(weights.ndim - 1))) * zero_point
    return sums

def quantize_biases(biases, scale_bias, zero_point_sum):
    """ Quantize the biases """
    quantized = np.round(biases * scale_bias) - zero_point_sum # Compute Sum wi * zp_i ahead of time!
    quantized = quantized.astype(np.int32) # Clip to int32 range  
    return quantized


In [24]:
#########################################
# Save Values to binary files #
#########################################

def output_to_bin(filename, path, flattened_data):
    """ Save the flattened data to a binary file """
    full_path = os.path.join(img_dir, path)
    full_path = os.path.join(full_path, filename + ".bin")
    flattened_data.tofile(full_path)
    #print(f"Saved {filename} to {full_path}")
    return

In [25]:
#########################################
# Quantize Images and Save to Binary Files #
#########################################

img_dir = os.path.abspath('img_data')
pathlib.Path(img_dir).mkdir(exist_ok=True)
plot_dir = os.path.abspath('plots')
pathlib.Path(plot_dir).mkdir(exist_ok=True) 

for subdir in ['quant_val', 'quant_weights', 'layer_outputs']:
    subdir_path = os.path.join(img_dir, subdir)
    if not os.path.exists(subdir_path):
        Path(subdir_path).mkdir()


In [26]:
############################################
# Do image inputs #
###########################################

activation_scales = []
zero_points = []
weight_scales = []

#weight_scales.append(1.0) # Input layer has no weights

flat_images = []
for i, img_data in enumerate(val_imgs):
    img = tf.cast(img_data['image'], tf.float32) / 255.0  # Normalize to [0, 1]
    img = img[0]
    #print(f"Processing image {i} with shape {img.shape}")
    #img = img.numpy().transpose(2,0,1)  # Change to CHW for easier flattening
    flat_img = img.numpy().flatten()
    #print(f"Processing image {i} with shape {img.shape}")
    flat_images.append(flat_img)

flat_flat_images = np.concatenate(flat_images)

image_average = get_average(flat_flat_images)

image_max_diff = get_max_difference(flat_flat_images, image_average)

activation_scales.append(get_input_scale(image_max_diff))

zero_points.append(get_input_zero_point(activation_scales[0], image_average))

print(f"Image Quantization Parameters: scale {activation_scales[0]}, zero_point {zero_points[0]}")
print(f"Image average {image_average}, max_difference {image_max_diff}")
print(f"All Activations Len: {len(all_activations)}")

# Quantize and save images
quantized_images = []

for i, image in enumerate(flat_images):
    quantized_image = quantize_inputs(image, activation_scales[0], zero_points[0])
    quantized_images.append(quantized_image)
    output_to_bin(f"image_{i}", "quant_val", quantized_image)

# # print one non-quantized image as a grid/image
# plt.imshow(flat_images[0].reshape(64, 64, 3), cmap='gray')
# plt.title("Non-Quantized Image 0")
# plt.show()

# #print one quantized image as a grid/image, dequantize first
# dequantized_image = (quantized_images[0].astype(np.float32) - zero_points[0]) / activation_scales[0]
# plt.imshow(dequantized_image.reshape(64, 64, 3), cmap='gray')
# plt.title("Quantized Image 0")
# plt.show()


Image Quantization Parameters: scale 1.7896433280097006, zero_point -1
Image average 0.4412294425660108, max_difference 0.5587705574339892
All Activations Len: 12


In [9]:
for i, layer_name in enumerate([layer.name for layer in model.layers]):
    print(f"Processing layer {i}: {layer_name} with activation shape {all_activations[i].shape}")
    layer = model.get_layer(layer_name)
    layer_activation = all_activations[i]
    first_image_activation = layer_activation[0] # CHANGE BACK TO 0 AFTER TESTING!
    #print(f"First image activation shape: {first_image_activation.shape}")
    #layer_activation = layer_activation.transpose(0,3,1,2)  # Change to NCHW for easier flattening if needed
    #first_image_activation = first_image_activation.transpose(2,0,1)  # Change to CHW for easier flattening if needed
    #print(f"First image activation shape after transpose: {first_image_activation.shape}")


    if 'conv' in layer_name or 'dense' in layer_name:
        # Input quantization parameters (from previous layer's output)
        input_scale = activation_scales[-1]
        input_zp = zero_points[-1]
        #input_weight_scale = weight_scales[-1]

        # 3. get weights and bias parameters
        weights = layer.get_weights()[0]  # Get weights
        max_weight = get_max_weight(weights)
        weight_scale = get_weight_scale(max_weight)
        weight_scales.append(weight_scale)
        quantized_weights = quantize_weights(weights, weight_scale)

        wi_zp_sum = compute_wi_zp_sum(quantized_weights, input_zp)

        biases = layer.get_weights()[1] if len(layer.get_weights()) > 1 else np.zeros(layer.output_shape[-1])
        print(f"debug: Layer {layer_name} input_scale {input_scale}, input_zp {input_zp}, weight_scale {weight_scale}, wi_zp_sum {wi_zp_sum}, average quantized weight {get_average(quantized_weights.flatten())}, max weight {max_weight}")
        bias_scale = get_bias_scale(input_scale, weight_scale)
        quantized_biases = quantize_biases(biases, bias_scale, wi_zp_sum)
        
        # <--- ADDED VISUALIZATION FOR WEIGHTS --->
        plot_histogram(quantized_weights,
                       f"Distribution of Quantized Weights - {layer_name}",
                       f"{layer_name}_weights_dist",
                       plot_dir,
                       bins=100,
                       data_range=(min_int_w, max_int_w))
        

        # <--- ADDED VISUALIZATION FOR BIASES --->
        plot_histogram(quantized_biases,
                       f"Distribution of Quantized Biases - {layer_name}",
                       f"{layer_name}_biases_dist",
                       plot_dir,
                       bins=100) # Let range be auto for int32
        
        plot_histogram(biases,
                       f"Distribution of Biases - {layer_name}",
                       f"{layer_name}_nonQuant_biases_dist",
                       plot_dir,
                       bins=100) # Let range be auto for int32


        #quantized_weights = quantized_weights.transpose(3,2,0,1)  # Change to OIHW for saving
        # Save quantized weights and biases
        output_to_bin(f"{layer_name}_weights", "quant_weights", quantized_weights.flatten())
        output_to_bin(f"{layer_name}_biases", "quant_weights", quantized_biases.flatten())
        print(f"Weight Scales {weight_scales}")
        print(f"Bias Scale {bias_scale}")
        # Now compute output quantization parameters for all layers
        flat_output = np.concatenate([layer_activation[j].flatten() for j in range(layer_activation.shape[0])])
        output_average = get_average(flat_output)
        output_max_diff = get_max_difference(flat_output, output_average)
        output_scale = get_input_scale(output_max_diff)
        output_zp = get_input_zero_point(output_scale, output_average)

        # Append the output params
        activation_scales.append(output_scale)
        zero_points.append(output_zp)
            # For max pooling and flatten, to match C++ approximation, quantize input, apply operation on int8, then use same params for output if same
    if 'pool' in layer_name or 'flatten' in layer_name:
        # Use input params for output to simulate int8 operation without requant
        output_scale = activation_scales[-1]
        output_zp = zero_points[-1]

        # Get previous activation (input to this layer)
        if i > 0:
            first_image_input = all_activations[i-1][0]
            input_shape = first_image_input.shape
            quantized_input = quantize_inputs(first_image_input, activation_scales[-1], zero_points[-1])

            if 'pool' in layer_name:
                # Implement max pooling on quantized input (assuming 2x2 pool, stride 2)
                h, w, c = input_shape
                q_input_reshaped = quantized_input.reshape(h, w, c)
                pooled_h, pooled_w = h//2, w//2
                quantized_pooled = np.zeros((pooled_h, pooled_w, c), dtype=np.int8)
                for ph in range(pooled_h):
                    for pw in range(pooled_w):
                        for pc in range(c):
                            patch = q_input_reshaped[ph*2:ph*2+2, pw*2:pw*2+2, pc]
                            quantized_pooled[ph, pw, pc] = np.max(patch)
                quantized_activation = quantized_pooled
            elif 'flatten' in layer_name:
                # Flatten doesn't change values
                quantized_activation = quantized_input.reshape(-1)

        else:
            # Should not happen
            quantized_activation = quantize_inputs(first_image_activation, activation_scales[-1], output_zp)

    else:
        # For other layers, quantize the output directly
        quantized_activation = quantize_inputs(first_image_activation, activation_scales[-1], output_zp)



    # <--- ADDED VISUALIZATION FOR ACTIVATIONS --->
    plot_histogram(quantized_activation,
                   f"Distribution of Quantized Activations (Image 0) - {layer_name}",
                   f"layer_{i}_{layer_name}_activation_dist",
                   plot_dir,
                   bins=100,
                   data_range=(min_int_a, max_int_a))
        # <--- ADDED VISUALIZATION FOR ACTIVATIONS --->
    plot_histogram(first_image_activation,
                   f"Distribution of Pre-Quantized Activations (Image 0) - {layer_name}",
                   f"layer_{i}_{layer_name}_nonquant_activation_dist",
                   plot_dir,
                   bins=100)
    
    # Save quantized activations
    output_to_bin(f"layer_{i}_output", "layer_outputs", quantized_activation.flatten())

    # Dequantize and compare error
    original_flat = first_image_activation.flatten()
    quant_flat = quantized_activation.flatten()
    dequant_flat = (quant_flat.astype(np.float32) - output_zp) / output_scale
    mae = np.mean(np.abs(original_flat - dequant_flat))
    mse = np.mean((original_flat - dequant_flat)**2)
    max_err = np.max(np.abs(original_flat - dequant_flat))
    print(f"Layer {i} {layer_name} - MAE: {mae}, MSE: {mse}, Max Error: {max_err}")
    plot_histogram(dequant_flat,
                    f"Distribution of De-Quantized Activations (Image 0) - {layer_name}",
                    f"layer_{i}_{layer_name}_dequant_activation_dist",
                    plot_dir,
                    bins=100)

# output the final layer output to bin (non quantized!)
output_to_bin("layer_11_output", "layer_outputs", all_activations[-1][0])
print(f"Activation Scales: {activation_scales}")
print(f"Weight Scales: {weight_scales}")
print(f"Zero Points: {zero_points}")

Processing layer 0: conv2d with activation shape (1000, 60, 60, 32)
debug: Layer conv2d input_scale 12.527503296067906, input_zp -6, weight_scale 23.111511869836942, wi_zp_sum [ 30  18  54 -18 108 264  36 420  12  96  18  12  24  -6   6   0  60  36
 510 -42 228 144   0 150 234   0 -24 258  12 120 -30 -60], average quantized weight -0.18541666666666667, max weight 0.30287936329841614
Saved plot: /home/tbibus/CprE587/CPRE-4870/Lab4/sw/cpre487-587-dnn-framework-main/python/plots/conv2d_weights_dist.png
Saved plot: /home/tbibus/CprE587/CPRE-4870/Lab4/sw/cpre487-587-dnn-framework-main/python/plots/conv2d_biases_dist.png
Saved plot: /home/tbibus/CprE587/CPRE-4870/Lab4/sw/cpre487-587-dnn-framework-main/python/plots/conv2d_nonQuant_biases_dist.png
Weight Scales [23.111511869836942]
Bias Scale 289.5295411264948
Saved plot: /home/tbibus/CprE587/CPRE-4870/Lab4/sw/cpre487-587-dnn-framework-main/python/plots/layer_0_conv2d_activation_dist.png
Saved plot: /home/tbibus/CprE587/CPRE-4870/Lab4/sw/cpre4

In [10]:
numMac = 0
for i, layer_name in enumerate([layer.name for layer in model.layers]):
    print(f"Processing layer {i}: {layer_name} with activation shape {all_activations[i].shape}")
    layer = model.get_layer(layer_name)
    layer_activation = all_activations[i]
    first_image_activation = layer_activation[0] # CHANGE BACK TO 0 AFTER TESTING!
    quantized_activation = quantize_inputs(first_image_activation, activation_scales[numMac], output_zp)
    if 'conv' in layer_name or 'dense' in layer_name:
        numMac += 1
    # Save quantized activations
    output_to_bin(f"layer_{i}_output", "layer_outputs", quantized_activation.flatten())


# output the final layer output to bin (non quantized!)
output_to_bin("layer_11_output", "layer_outputs", all_activations[-1][0])
print(f"Activation Scales: {activation_scales}")
print(f"Weight Scales: {weight_scales}")
print(f"Zero Points: {zero_points}")

Processing layer 0: conv2d with activation shape (1000, 60, 60, 32)
Processing layer 1: conv2d_1 with activation shape (1000, 56, 56, 32)
Processing layer 2: max_pooling2d with activation shape (1000, 28, 28, 32)
Processing layer 3: conv2d_2 with activation shape (1000, 26, 26, 64)
Processing layer 4: conv2d_3 with activation shape (1000, 24, 24, 64)
Processing layer 5: max_pooling2d_1 with activation shape (1000, 12, 12, 64)
Processing layer 6: conv2d_4 with activation shape (1000, 10, 10, 64)
Processing layer 7: conv2d_5 with activation shape (1000, 8, 8, 128)
Processing layer 8: max_pooling2d_2 with activation shape (1000, 4, 4, 128)
Processing layer 9: flatten with activation shape (1000, 2048)
Processing layer 10: dense with activation shape (1000, 256)
Processing layer 11: dense_1 with activation shape (1000, 200)
Activation Scales: [12.527503296067906, 4.140228981783817, 1.71639332572974, 2.148475256830365, 1.8969632632023494, 1.340582357644912, 0.7848926509614248, 0.45321418509

In [11]:
for i, layer_name in enumerate([layer.name for layer in model.layers]):
    print(f"Processing layer {i}: {layer_name} with activation shape {all_activations[i].shape}")
    layer_activation = all_activations[i]
    print(f"Maximum activation value: {np.max(layer_activation)}")
    print(f"Minimum activation value: {np.min(layer_activation)}")
    print(f"average activation value: {np.mean(layer_activation)}")

print("### Model Complexity Analysis (MACs) ###\n")
total_macs = 0

for i, layer_name in enumerate([layer.name for layer in model.layers]):
    # Get the actual layer object from the model
    layer = model.layers[i]
    
    # --- Original user code (logging) ---
    print(f"Processing layer {i}: {layer_name} with activation shape {all_activations[i].shape}")
    layer_activation = all_activations[i]
    print(f"Maximum activation value: {np.max(layer_activation)}")
    print(f"Minimum activation value: {np.min(layer_activation)}")
    print(f"average activation value: {np.mean(layer_activation)}")

    # --- Robust MACs calculation (use weight shapes when possible) ---
    macs = 0  # Default for layers with no MACs (Pooling, Flatten, Activation, etc.)

    try:
        # Conv2D
        if isinstance(layer, tf.keras.layers.Conv2D):
            weights = layer.get_weights()
            if weights:
                w = weights[0]  # shape: (K_h, K_w, C_in, C_out)
                K_h, K_w, C_in = int(w.shape[0]), int(w.shape[1]), int(w.shape[2])
                C_out = int(w.shape[3])
            else:
                # fallback to layer attributes where available
                K_h, K_w = layer.kernel_size
                C_in = getattr(layer, 'input_shape', (None, None, None, None))[-1] or 0
                C_out = getattr(layer, 'filters', 0)

            H_out = int(all_activations[i].shape[1])
            W_out = int(all_activations[i].shape[2])

            macs = (K_h * K_w * C_in) * (H_out * W_out * C_out)

        # Dense
        elif isinstance(layer, tf.keras.layers.Dense):
            weights = layer.get_weights()
            if weights:
                w = weights[0]  # shape: (N_in, N_out)
                N_in, N_out = int(w.shape[0]), int(w.shape[1])
            else:
                N_in = getattr(layer, 'input_shape', (None,))[-1] or 0
                N_out = getattr(layer, 'units', 0)

            macs = N_in * N_out

        # DepthwiseConv2D
        elif isinstance(layer, tf.keras.layers.DepthwiseConv2D):
            weights = layer.get_weights()
            if weights:
                w = weights[0]  # shape: (K_h, K_w, C_in, depth_multiplier)
                K_h, K_w, C_in = int(w.shape[0]), int(w.shape[1]), int(w.shape[2])
                depth_mul = int(w.shape[3])
            else:
                K_h, K_w = layer.kernel_size
                C_in = getattr(layer, 'input_shape', (None, None, None, None))[-1] or 0
                depth_mul = getattr(layer, 'depth_multiplier', 1)

            H_out = int(all_activations[i].shape[1])
            W_out = int(all_activations[i].shape[2])

            # Depthwise MACs include the depth multiplier
            macs = (K_h * K_w * C_in * depth_mul) * (H_out * W_out)

        # Add other layer types with MACs here if needed...

        # Ensure we don't add NaN (use numpy here; avoids needing math import)
        if not np.isnan(macs):
            total_macs += macs

        print(f"Number of MACs: {macs:,.0f}")

    except TypeError:
        # This catch block handles cases where a shape dimension is None
        # (e.g., model(inputs=...) was not called with a defined batch size)
        print("Number of MACs: N/A (Dynamic Shape Detected)")
    except Exception as e:
        # Catch any unexpected errors and report
        print(f"Number of MACs: N/A (Error: {e})")
    
    print("---")  # Separator

print(f"\nTotal Model MACs (for a single forward pass): {total_macs:,.0f}")

Processing layer 0: conv2d with activation shape (1000, 60, 60, 32)
Maximum activation value: 1.7239302396774292
Minimum activation value: 0.0
average activation value: 0.03320247307419777
Processing layer 1: conv2d_1 with activation shape (1000, 56, 56, 32)
Maximum activation value: 4.109264373779297
Minimum activation value: 0.0
average activation value: 0.03094499371945858
Processing layer 2: max_pooling2d with activation shape (1000, 28, 28, 32)
Maximum activation value: 4.109264373779297
Minimum activation value: 0.0
average activation value: 0.05215341970324516
Processing layer 3: conv2d_2 with activation shape (1000, 26, 26, 64)
Maximum activation value: 3.291311025619507
Minimum activation value: 0.0
average activation value: 0.033186521381139755
Processing layer 4: conv2d_3 with activation shape (1000, 24, 24, 64)
Maximum activation value: 3.731480360031128
Minimum activation value: 0.0
average activation value: 0.04137197881937027
Processing layer 5: max_pooling2d_1 with acti

## THIS IS WERE WE GET 1000 INFERENCES AND COMPARE!!!

In [32]:
# Assuming 'bits', 'ds_val' are defined, and 'numpy' and 'pathlib' are imported:
# import numpy as np
# from pathlib import Path
print("### Calculating Top-1, Top-5, Top-10 Accuracy ###")
print(f"Using softmax data from: img_data/{bits}bitSoftmax")
data_dir = Path(f"img_data/{bits}bitSoftmax")
num_images = 1000
top1 = 0
top5 = 0
top10 = 0
labels = []

# This outer loop gets the batch of labels
for batch in ds_val.batch(1000).take(1):
    labels = batch['label'].numpy()
    assert len(labels) == 1000
    
    # This inner loop processes each image in the batch
    for i in range(num_images):
        softmax_path = data_dir / f"softmax_output_{i}.bin"
        if not softmax_path.exists():
            print(f"Missing {softmax_path}")
            continue
        
        softmax = np.fromfile(softmax_path, dtype=np.float32)
        #assert len(softmax) == 256

        # --- THIS IS THE BLOCK THAT WAS MOVED ---
        # This logic must be INSIDE the loop to run for each image 'i'
        top_indices = np.argsort(softmax)[::-1]
        true_label = labels[i] # Get the label for the current image
        
        if true_label == top_indices[0]:
            top1 += 1
        if true_label in top_indices[:5]:
            top5 += 1
        if true_label in top_indices[:10]:
            top10 += 1
        # --- END OF MOVED BLOCK ---

# These print statements run once at the very end,
# after all images have been processed.
print(f"Top-1 Accuracy: {top1 / num_images * 100:.2f}%")
print(f"Top-5 Accuracy: {top5 / num_images * 100:.2f}%")
print(f"Top-10 Accuracy: {top10 / num_images * 100:.2f}%")

### Calculating Top-1, Top-5, Top-10 Accuracy ###
Using softmax data from: img_data/4bitSoftmax


2025-10-29 23:46:42.023719: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


Top-1 Accuracy: 1.30%
Top-5 Accuracy: 4.50%
Top-10 Accuracy: 8.00%


2025-10-29 23:46:42.878445: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
